In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd


N_OUTER_FOLDS = 5
TARGET_LOG_COL = "Target_Log"
WASTE_COLS = ["Is_AW", "Is_CDW", "Is_IW", "Is_MSW"]
WASTE_NAME_MAP = {"Is_AW": "AW", "Is_CDW": "CDW", "Is_IW": "IW", "Is_MSW": "MSW"}
FORMAL_TEN_FEATURE_CONTRACT = [
    "NY.GDP.MKTP.PP.KD",
    "SP.POP.TOTL",
    "NY.GDP.PCAP.PP.KD",
    "EN.POP.DNST",
    "Proxy_NY.GDP.MKTP.PP.KD_log1p",
    "Proxy_SP.POP.TOTL_log1p",
    "Proxy_NY.GDP.PCAP.PP.KD_log1p",
    "Proxy_EN.POP.DNST_log1p",
    "EN.GHG.CO2.MT.CE.AR5",
    "Proxy_EN.GHG.CO2.MT.CE.AR5_log1p",
]


def find_release_root(start: Path) -> Path:
    sentinel_dirs = ("Step0_Data", "Step4_ModelInputs")
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        release_candidate = candidate / "GSW_Clean_Notebook_Release"
        if all((release_candidate / name).is_dir() for name in sentinel_dirs):
            return release_candidate
    raise FileNotFoundError("Could not locate standalone release root")


def _mode_or_first(series: pd.Series):
    non_null = series.dropna()
    if non_null.empty:
        return np.nan
    modes = non_null.mode(dropna=True)
    if not modes.empty:
        return modes.iloc[0]
    return non_null.iloc[0]


def derive_waste_flag(df: pd.DataFrame) -> pd.Series:
    flags_raw = df[WASTE_COLS].apply(pd.to_numeric, errors="coerce")
    flags = flags_raw.fillna(0.0)
    all_nan_mask = flags_raw.isna().all(axis=1)
    all_zero_mask = flags.eq(0.0).all(axis=1)
    tied_max_mask = flags.eq(flags.max(axis=1), axis=0).sum(axis=1).gt(1) & ~all_zero_mask

    if all_nan_mask.any():
        bad_idx = all_nan_mask[all_nan_mask].index.tolist()[:5]
        raise ValueError(f"derive_waste_flag found all NaN waste indicators at rows {bad_idx}")
    if all_zero_mask.any():
        bad_idx = all_zero_mask[all_zero_mask].index.tolist()[:5]
        raise ValueError(f"derive_waste_flag found all zero waste indicators at rows {bad_idx}")
    if tied_max_mask.any():
        bad_idx = tied_max_mask[tied_max_mask].index.tolist()[:5]
        raise ValueError(f"derive_waste_flag found tied max waste indicators at rows {bad_idx}")

    idx = flags.to_numpy(dtype=float).argmax(axis=1)
    names = np.array([WASTE_NAME_MAP[c] for c in WASTE_COLS], dtype=object)
    return pd.Series(names[idx], index=df.index, name="WasteFlag")


def load_inputs(longtable_path: Path) -> pd.DataFrame:
    long_df = pd.read_csv(longtable_path).copy()
    long_df["Year"] = pd.to_numeric(long_df["Year"], errors="coerce").astype("Int64")
    long_df["Waste_Generation_t"] = pd.to_numeric(long_df["Waste_Generation_t"], errors="coerce")
    long_df = long_df.dropna(subset=["Country Code", "Year", "Waste_Generation_t"]).copy()
    long_df["Country Code"] = long_df["Country Code"].astype(str).str.strip()
    long_df = long_df.loc[long_df["Country Code"] != ""].copy()
    long_df["Year"] = long_df["Year"].astype(int)
    long_df["WasteFlag"] = derive_waste_flag(long_df)
    long_df[TARGET_LOG_COL] = np.log1p(long_df["Waste_Generation_t"].clip(lower=0.0))
    return long_df


def build_country_profile(long_df: pd.DataFrame) -> pd.DataFrame:
    d = long_df.copy()
    d["log_wg"] = np.log1p(d["Waste_Generation_t"].clip(lower=0.0))

    base = (
        d.groupby("Country Code", dropna=False)
        .agg(
            **{
                "Country Name": ("Country Name", _mode_or_first),
                "IncomeGroup": ("IncomeGroup", _mode_or_first),
                "Region": ("Region", _mode_or_first),
                "n_rows": ("Year", "count"),
                "n_years": ("Year", "nunique"),
                "year_min": ("Year", "min"),
                "year_max": ("Year", "max"),
                "wg_sum": ("Waste_Generation_t", "sum"),
                "wg_mean": ("Waste_Generation_t", "mean"),
                "wg_median": ("Waste_Generation_t", "median"),
                "wg_std": ("Waste_Generation_t", "std"),
                "log_wg_mean": ("log_wg", "mean"),
                "log_wg_median": ("log_wg", "median"),
            }
        )
        .reset_index()
    )
    base["wg_std"] = base["wg_std"].fillna(0.0)

    for col in WASTE_COLS:
        ratio = d.groupby("Country Code", dropna=False)[col].mean().rename(f"{WASTE_NAME_MAP[col].lower()}_ratio")
        base = base.merge(ratio.reset_index(), on="Country Code", how="left")

    ratio_cols = ["aw_ratio", "cdw_ratio", "iw_ratio", "msw_ratio"]
    base[ratio_cols] = base[ratio_cols].fillna(0.0)
    dom_idx = base[ratio_cols].to_numpy(dtype=float).argmax(axis=1)
    dom_names = np.array(["AW", "CDW", "IW", "MSW"], dtype=object)
    base["dominant_waste"] = dom_names[dom_idx]
    return base


def _safe_rel_diff(x: float, target: float) -> float:
    den = max(abs(target), 1e-12)
    return (x - target) / den


def assign_folds_greedy(profile_df: pd.DataFrame, n_splits: int = N_OUTER_FOLDS) -> pd.DataFrame:
    d = profile_df.copy()
    d["IncomeGroup"] = d["IncomeGroup"].astype(str)
    d["n_rows"] = pd.to_numeric(d["n_rows"], errors="coerce").fillna(0.0).astype(float)
    d["log_wg_mean"] = pd.to_numeric(d["log_wg_mean"], errors="coerce").fillna(0.0).astype(float)
    for c in ["aw_ratio", "cdw_ratio", "iw_ratio", "msw_ratio"]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0).astype(float)

    income_values = sorted(d["IncomeGroup"].unique().tolist())
    total_rows = float(d["n_rows"].sum())
    total_countries = float(len(d))
    target_rows_per_fold = total_rows / max(float(n_splits), 1.0)
    target_countries_per_fold = total_countries / max(float(n_splits), 1.0)
    income_targets = {
        inc: float((d["IncomeGroup"] == inc).sum()) / max(float(n_splits), 1.0)
        for inc in income_values
    }

    global_log_mean = float(np.average(d["log_wg_mean"], weights=np.maximum(d["n_rows"], 1.0)))
    global_aw = float(np.average(d["aw_ratio"], weights=np.maximum(d["n_rows"], 1.0)))
    global_cdw = float(np.average(d["cdw_ratio"], weights=np.maximum(d["n_rows"], 1.0)))
    global_iw = float(np.average(d["iw_ratio"], weights=np.maximum(d["n_rows"], 1.0)))
    global_msw = float(np.average(d["msw_ratio"], weights=np.maximum(d["n_rows"], 1.0)))

    fold_stats: List[Dict] = []
    for _ in range(n_splits):
        fold_stats.append(
            {
                "rows": 0.0,
                "countries": 0.0,
                "log_w_sum": 0.0,
                "aw_sum": 0.0,
                "cdw_sum": 0.0,
                "iw_sum": 0.0,
                "msw_sum": 0.0,
                "income": {k: 0.0 for k in income_values},
            }
        )

    assignments: List[Dict] = []

    for inc in income_values:
        group_df = d.loc[d["IncomeGroup"] == inc].copy()
        group_df = group_df.sort_values(["n_rows", "Country Code"], ascending=[False, True]).reset_index(drop=True)
        inc_target = income_targets[inc]

        for _, row in group_df.iterrows():
            r_rows = float(row["n_rows"])
            r_log = float(row["log_wg_mean"])
            r_aw = float(row["aw_ratio"])
            r_cdw = float(row["cdw_ratio"])
            r_iw = float(row["iw_ratio"])
            r_msw = float(row["msw_ratio"])

            min_inc_count = min(fold_stats[k]["income"][inc] for k in range(n_splits))
            candidate_folds = [
                k
                for k in range(n_splits)
                if fold_stats[k]["income"][inc] <= (min_inc_count + 1.0)
            ]
            if len(candidate_folds) == 0:
                candidate_folds = list(range(n_splits))

            best_fold = candidate_folds[0]
            best_score = None
            for fold_id in candidate_folds:
                fs = fold_stats[fold_id]
                rows_new = fs["rows"] + r_rows
                countries_new = fs["countries"] + 1.0

                log_mean_new = (fs["log_w_sum"] + r_log * r_rows) / max(rows_new, 1e-12)
                aw_mean_new = (fs["aw_sum"] + r_aw * r_rows) / max(rows_new, 1e-12)
                cdw_mean_new = (fs["cdw_sum"] + r_cdw * r_rows) / max(rows_new, 1e-12)
                iw_mean_new = (fs["iw_sum"] + r_iw * r_rows) / max(rows_new, 1e-12)
                msw_mean_new = (fs["msw_sum"] + r_msw * r_rows) / max(rows_new, 1e-12)

                row_cost = _safe_rel_diff(rows_new, target_rows_per_fold) ** 2
                country_cost = _safe_rel_diff(countries_new, target_countries_per_fold) ** 2

                weak_shape_cost = 0.0
                weak_shape_cost += (log_mean_new - global_log_mean) ** 2
                weak_shape_cost += (aw_mean_new - global_aw) ** 2
                weak_shape_cost += (cdw_mean_new - global_cdw) ** 2
                weak_shape_cost += (iw_mean_new - global_iw) ** 2
                weak_shape_cost += (msw_mean_new - global_msw) ** 2

                inc_new = fs["income"][inc] + 1.0
                inc_cost = _safe_rel_diff(inc_new, inc_target) ** 2
                inc_counts_after = [
                    fold_stats[k]["income"][inc] + (1.0 if k == fold_id else 0.0)
                    for k in range(n_splits)
                ]
                inc_spread_after = float(max(inc_counts_after) - min(inc_counts_after))
                inc_spread_penalty = max(0.0, inc_spread_after - 1.0) ** 2

                score = (
                    16.0 * row_cost
                    + 2.0 * country_cost
                    + 3.0 * inc_cost
                    + 8.0 * inc_spread_penalty
                    + 0.2 * weak_shape_cost
                )
                tie = (score, inc_spread_after, fs["rows"], fold_id)
                if best_score is None or tie < best_score:
                    best_score = tie
                    best_fold = fold_id

            fs = fold_stats[best_fold]
            fs["rows"] += r_rows
            fs["countries"] += 1.0
            fs["log_w_sum"] += r_log * r_rows
            fs["aw_sum"] += r_aw * r_rows
            fs["cdw_sum"] += r_cdw * r_rows
            fs["iw_sum"] += r_iw * r_rows
            fs["msw_sum"] += r_msw * r_rows
            fs["income"][inc] += 1.0
            assignments.append({"Country Code": row["Country Code"], "outer_fold": int(best_fold + 1)})

    out = pd.DataFrame(assignments)
    meta = d[["Country Code", "IncomeGroup", "n_rows", "log_wg_mean", "aw_ratio", "cdw_ratio", "iw_ratio", "msw_ratio"]].copy()

    def _fold_shape_contrib(rows_v: float, log_sum_v: float, aw_sum_v: float, cdw_sum_v: float, iw_sum_v: float, msw_sum_v: float) -> float:
        den = max(rows_v, 1e-12)
        log_mean_v = log_sum_v / den
        aw_mean_v = aw_sum_v / den
        cdw_mean_v = cdw_sum_v / den
        iw_mean_v = iw_sum_v / den
        msw_mean_v = msw_sum_v / den
        c = 0.0
        c += (log_mean_v - global_log_mean) ** 2
        c += (aw_mean_v - global_aw) ** 2
        c += (cdw_mean_v - global_cdw) ** 2
        c += (iw_mean_v - global_iw) ** 2
        c += (msw_mean_v - global_msw) ** 2
        return float(c)

    shape_budget = None
    for _ in range(3000):
        merged = out.merge(meta, on="Country Code", how="left", validate="1:1")
        row_sum = merged.groupby("outer_fold")["n_rows"].sum().reindex(range(1, n_splits + 1), fill_value=0.0)
        src_fold = int(row_sum.idxmax())
        dst_fold = int(row_sum.idxmin())
        cur_gap = float(row_sum[src_fold] - row_sum[dst_fold])
        if cur_gap <= 1.0:
            break

        merged["w_log"] = merged["log_wg_mean"] * merged["n_rows"]
        merged["w_aw"] = merged["aw_ratio"] * merged["n_rows"]
        merged["w_cdw"] = merged["cdw_ratio"] * merged["n_rows"]
        merged["w_iw"] = merged["iw_ratio"] * merged["n_rows"]
        merged["w_msw"] = merged["msw_ratio"] * merged["n_rows"]

        fold_state = (
            merged.groupby("outer_fold", dropna=False)
            .agg(
                rows=("n_rows", "sum"),
                log_sum=("w_log", "sum"),
                aw_sum=("w_aw", "sum"),
                cdw_sum=("w_cdw", "sum"),
                iw_sum=("w_iw", "sum"),
                msw_sum=("w_msw", "sum"),
            )
            .reindex(range(1, n_splits + 1), fill_value=0.0)
        )

        contrib = {
            fold: _fold_shape_contrib(
                float(fold_state.loc[fold, "rows"]),
                float(fold_state.loc[fold, "log_sum"]),
                float(fold_state.loc[fold, "aw_sum"]),
                float(fold_state.loc[fold, "cdw_sum"]),
                float(fold_state.loc[fold, "iw_sum"]),
                float(fold_state.loc[fold, "msw_sum"]),
            )
            for fold in range(1, n_splits + 1)
        }
        cur_shape = float(sum(contrib.values()))
        if shape_budget is None:
            shape_budget = max(cur_shape * 1.25, cur_shape + 1e-12)

        income_tab = pd.crosstab(merged["outer_fold"], merged["IncomeGroup"]).reindex(range(1, n_splits + 1), fill_value=0.0)

        best_swap = None
        best_swap_gap = cur_gap
        for inc in income_values:
            src_pool = merged.loc[(merged["outer_fold"] == src_fold) & (merged["IncomeGroup"] == inc)].sort_values("n_rows", ascending=False)
            dst_pool = merged.loc[(merged["outer_fold"] == dst_fold) & (merged["IncomeGroup"] == inc)].sort_values("n_rows", ascending=True)
            if len(src_pool) == 0 or len(dst_pool) == 0:
                continue
            for _, src_row in src_pool.iterrows():
                for _, dst_row in dst_pool.iterrows():
                    src_rows = float(src_row["n_rows"])
                    dst_rows = float(dst_row["n_rows"])
                    if src_rows <= dst_rows:
                        continue
                    src_state = fold_state.loc[src_fold]
                    dst_state = fold_state.loc[dst_fold]

                    src_rows_new = float(src_state["rows"] - src_rows + dst_rows)
                    dst_rows_new = float(dst_state["rows"] - dst_rows + src_rows)
                    src_log_sum_new = float(src_state["log_sum"] - src_row["w_log"] + dst_row["w_log"])
                    dst_log_sum_new = float(dst_state["log_sum"] - dst_row["w_log"] + src_row["w_log"])
                    src_aw_sum_new = float(src_state["aw_sum"] - src_row["w_aw"] + dst_row["w_aw"])
                    dst_aw_sum_new = float(dst_state["aw_sum"] - dst_row["w_aw"] + src_row["w_aw"])
                    src_cdw_sum_new = float(src_state["cdw_sum"] - src_row["w_cdw"] + dst_row["w_cdw"])
                    dst_cdw_sum_new = float(dst_state["cdw_sum"] - dst_row["w_cdw"] + src_row["w_cdw"])
                    src_iw_sum_new = float(src_state["iw_sum"] - src_row["w_iw"] + dst_row["w_iw"])
                    dst_iw_sum_new = float(dst_state["iw_sum"] - dst_row["w_iw"] + src_row["w_iw"])
                    src_msw_sum_new = float(src_state["msw_sum"] - src_row["w_msw"] + dst_row["w_msw"])
                    dst_msw_sum_new = float(dst_state["msw_sum"] - dst_row["w_msw"] + src_row["w_msw"])

                    src_contrib_new = _fold_shape_contrib(
                        src_rows_new,
                        src_log_sum_new,
                        src_aw_sum_new,
                        src_cdw_sum_new,
                        src_iw_sum_new,
                        src_msw_sum_new,
                    )
                    dst_contrib_new = _fold_shape_contrib(
                        dst_rows_new,
                        dst_log_sum_new,
                        dst_aw_sum_new,
                        dst_cdw_sum_new,
                        dst_iw_sum_new,
                        dst_msw_sum_new,
                    )
                    new_shape = float(cur_shape - contrib[src_fold] - contrib[dst_fold] + src_contrib_new + dst_contrib_new)
                    if new_shape > shape_budget:
                        continue

                    new_gap = abs((row_sum[src_fold] - src_rows + dst_rows) - (row_sum[dst_fold] - dst_rows + src_rows))
                    candidate = (float(new_gap), new_shape)
                    best = None if best_swap is None else (best_swap_gap, best_swap[2])
                    if best is None or candidate < best:
                        best_swap_gap = float(new_gap)
                        best_swap = (src_row["Country Code"], dst_row["Country Code"], float(new_shape))

        if best_swap is not None:
            src_country, dst_country, _ = best_swap
            out.loc[out["Country Code"] == src_country, "outer_fold"] = dst_fold
            out.loc[out["Country Code"] == dst_country, "outer_fold"] = src_fold
            continue

        src_candidates = merged.loc[merged["outer_fold"] == src_fold].sort_values("n_rows", ascending=True)

        move_country = None
        move_best_gap = cur_gap
        move_shape = cur_shape
        for _, cand in src_candidates.iterrows():
            inc = str(cand["IncomeGroup"])
            c_rows = float(cand["n_rows"])
            src_inc = float(income_tab.loc[src_fold, inc])
            dst_inc = float(income_tab.loc[dst_fold, inc])
            inc_target = income_targets[inc]

            src_new = src_inc - 1.0
            dst_new = dst_inc + 1.0
            if abs(src_new - inc_target) > 1.0 or abs(dst_new - inc_target) > 1.0:
                continue

            col_counts = income_tab[inc].astype(float).copy()
            col_counts.loc[src_fold] = src_new
            col_counts.loc[dst_fold] = dst_new
            if float(col_counts.max() - col_counts.min()) > 1.0:
                continue

            src_state = fold_state.loc[src_fold]
            dst_state = fold_state.loc[dst_fold]

            src_rows_new = float(src_state["rows"] - c_rows)
            dst_rows_new = float(dst_state["rows"] + c_rows)
            src_log_sum_new = float(src_state["log_sum"] - cand["w_log"])
            dst_log_sum_new = float(dst_state["log_sum"] + cand["w_log"])
            src_aw_sum_new = float(src_state["aw_sum"] - cand["w_aw"])
            dst_aw_sum_new = float(dst_state["aw_sum"] + cand["w_aw"])
            src_cdw_sum_new = float(src_state["cdw_sum"] - cand["w_cdw"])
            dst_cdw_sum_new = float(dst_state["cdw_sum"] + cand["w_cdw"])
            src_iw_sum_new = float(src_state["iw_sum"] - cand["w_iw"])
            dst_iw_sum_new = float(dst_state["iw_sum"] + cand["w_iw"])
            src_msw_sum_new = float(src_state["msw_sum"] - cand["w_msw"])
            dst_msw_sum_new = float(dst_state["msw_sum"] + cand["w_msw"])

            src_contrib_new = _fold_shape_contrib(
                src_rows_new,
                src_log_sum_new,
                src_aw_sum_new,
                src_cdw_sum_new,
                src_iw_sum_new,
                src_msw_sum_new,
            )
            dst_contrib_new = _fold_shape_contrib(
                dst_rows_new,
                dst_log_sum_new,
                dst_aw_sum_new,
                dst_cdw_sum_new,
                dst_iw_sum_new,
                dst_msw_sum_new,
            )
            new_shape = float(cur_shape - contrib[src_fold] - contrib[dst_fold] + src_contrib_new + dst_contrib_new)
            if new_shape > shape_budget:
                continue

            new_gap = abs((row_sum[src_fold] - c_rows) - (row_sum[dst_fold] + c_rows))
            candidate = (float(new_gap), float(new_shape))
            best = None if move_country is None else (move_best_gap, move_shape)
            if best is None or candidate < best:
                move_best_gap = float(new_gap)
                move_shape = float(new_shape)
                move_country = cand["Country Code"]

        if move_country is None:
            break
        out.loc[out["Country Code"] == move_country, "outer_fold"] = dst_fold

    return out


def enforce_income_waste_coverage(
    long_df: pd.DataFrame,
    fold_df: pd.DataFrame,
    n_splits: int = N_OUTER_FOLDS,
    max_iter: int = 2000,
) -> pd.DataFrame:
    required_long_cols = {"Country Code", "IncomeGroup", "WasteFlag"}
    missing_long_cols = sorted(required_long_cols.difference(long_df.columns))
    if missing_long_cols:
        raise ValueError(f"long_df missing required columns: {missing_long_cols}")

    required_fold_cols = {"Country Code", "outer_fold"}
    missing_fold_cols = sorted(required_fold_cols.difference(fold_df.columns))
    if missing_fold_cols:
        raise ValueError(f"fold_df missing required columns: {missing_fold_cols}")

    work_fold = fold_df[["Country Code", "outer_fold"]].copy()
    work_fold["Country Code"] = work_fold["Country Code"].astype(str)
    work_fold["outer_fold"] = pd.to_numeric(work_fold["outer_fold"], errors="coerce").astype("Int64")
    work_fold = work_fold.dropna(subset=["Country Code", "outer_fold"]).copy()
    work_fold["outer_fold"] = work_fold["outer_fold"].astype(int)
    if work_fold["Country Code"].duplicated().any():
        raise ValueError("fold_df has duplicated Country Code")

    d = long_df.copy()
    d["Country Code"] = d["Country Code"].astype(str)
    d["IncomeGroup"] = d["IncomeGroup"].astype(str)
    d["WasteFlag"] = d["WasteFlag"].astype(str)

    country_income = (
        d.groupby("Country Code", dropna=False)["IncomeGroup"]
        .agg(lambda s: s.mode(dropna=False).iloc[0] if len(s.mode(dropna=False)) > 0 else s.iloc[0])
        .rename("IncomeGroup")
        .reset_index()
    )
    country_rows = d.groupby("Country Code", dropna=False).size().rename("n_rows").reset_index()

    waste_values = sorted(d["WasteFlag"].dropna().astype(str).unique().tolist())
    waste_rows = (
        d.groupby(["Country Code", "WasteFlag"], dropna=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=waste_values, fill_value=0)
        .reset_index()
    )
    waste_row_cols = [f"w_rows_{w}" for w in waste_values]
    waste_rows.columns = ["Country Code"] + waste_row_cols

    country_meta = country_income.merge(country_rows, on="Country Code", how="left", validate="1:1")
    country_meta = country_meta.merge(waste_rows, on="Country Code", how="left", validate="1:1")
    for c in waste_row_cols:
        country_meta[c] = pd.to_numeric(country_meta[c], errors="coerce").fillna(0.0).astype(float)
    country_meta["n_rows"] = pd.to_numeric(country_meta["n_rows"], errors="coerce").fillna(0.0).astype(float)

    country_fold = work_fold.merge(country_meta, on="Country Code", how="left", validate="1:1")

    def _fold_row_sum(df: pd.DataFrame) -> pd.Series:
        return df.groupby("outer_fold", dropna=False)["n_rows"].sum().reindex(range(1, n_splits + 1), fill_value=0.0)

    def _income_cov(df: pd.DataFrame, income: str) -> pd.DataFrame:
        sub = df.loc[df["IncomeGroup"] == income].copy()
        cov = sub.groupby("outer_fold", dropna=False)[waste_row_cols].sum().reindex(range(1, n_splits + 1), fill_value=0.0)
        return cov

    def _is_safe_swap(cov: pd.DataFrame, target_fold: int, donor_fold: int, src_row: pd.Series, donor_row: pd.Series) -> bool:
        for w_col in waste_row_cols:
            t_new = float(cov.loc[target_fold, w_col] - float(src_row[w_col]) + float(donor_row[w_col]))
            d_new = float(cov.loc[donor_fold, w_col] - float(donor_row[w_col]) + float(src_row[w_col]))
            if t_new <= 0.0 or d_new <= 0.0:
                return False
        return True

    def _coverage_gaps(df: pd.DataFrame) -> List[str]:
        gaps: List[str] = []
        income_values = sorted(df["IncomeGroup"].dropna().astype(str).unique().tolist())
        for income in income_values:
            cov = _income_cov(df, income)
            for waste_flag, w_col in zip(waste_values, waste_row_cols):
                if float(cov[w_col].sum()) <= 0.0:
                    continue
                missing_folds = [str(fold) for fold in range(1, n_splits + 1) if float(cov.loc[fold, w_col]) <= 0.0]
                if missing_folds:
                    gaps.append(f"IncomeGroup={income}, WasteFlag={waste_flag}, missing_folds={','.join(missing_folds)}")
        return gaps

    for _ in range(max_iter):
        changed = False

        income_values = sorted(country_fold["IncomeGroup"].dropna().astype(str).unique().tolist())
        for income in income_values:
            cov = _income_cov(country_fold, income)

            for w_col in waste_row_cols:
                if float(cov[w_col].sum()) <= 0.0:
                    continue

                target_folds = [f for f in range(1, n_splits + 1) if float(cov.loc[f, w_col]) <= 0.0]
                for target_fold in target_folds:
                    src_pool = country_fold.loc[
                        (country_fold["IncomeGroup"] == income)
                        & (country_fold["outer_fold"] == target_fold)
                    ].copy()
                    if src_pool.empty:
                        continue
                    src_pool = src_pool.sort_values([w_col, "n_rows", "Country Code"], ascending=[True, False, True])

                    donor_pool = country_fold.loc[
                        (country_fold["IncomeGroup"] == income)
                        & (country_fold["outer_fold"] != target_fold)
                        & (country_fold[w_col] > 0.0)
                    ].copy()
                    if donor_pool.empty:
                        continue
                    donor_pool = donor_pool.sort_values([w_col, "n_rows", "Country Code"], ascending=[False, False, True])

                    row_sum = _fold_row_sum(country_fold)
                    best = None

                    for _, donor_row in donor_pool.iterrows():
                        donor_fold = int(donor_row["outer_fold"])
                        for _, src_row in src_pool.iterrows():
                            if not _is_safe_swap(cov, target_fold, donor_fold, src_row, donor_row):
                                continue

                            row_sum_new = row_sum.copy()
                            row_sum_new.loc[target_fold] = row_sum_new.loc[target_fold] - float(src_row["n_rows"]) + float(donor_row["n_rows"])
                            row_sum_new.loc[donor_fold] = row_sum_new.loc[donor_fold] - float(donor_row["n_rows"]) + float(src_row["n_rows"])
                            new_gap = float(row_sum_new.max() - row_sum_new.min())

                            score = (
                                new_gap,
                                abs(float(donor_row["n_rows"]) - float(src_row["n_rows"])),
                                -float(donor_row[w_col]),
                                str(src_row["Country Code"]),
                                str(donor_row["Country Code"]),
                            )
                            if best is None or score < best[0]:
                                best = (score, str(src_row["Country Code"]), str(donor_row["Country Code"]), donor_fold)

                    if best is None:
                        continue

                    _, src_country, donor_country, donor_fold = best
                    country_fold.loc[country_fold["Country Code"] == src_country, "outer_fold"] = donor_fold
                    country_fold.loc[country_fold["Country Code"] == donor_country, "outer_fold"] = target_fold
                    changed = True
                    cov = _income_cov(country_fold, income)

        if not changed:
            break

    out = country_fold[["Country Code", "outer_fold"]].copy()
    out = out.sort_values(["outer_fold", "Country Code"]).reset_index(drop=True)
    out["outer_fold"] = pd.to_numeric(out["outer_fold"], errors="coerce").astype(int)

    return out


def build_row_manifest(long_df: pd.DataFrame, fold_df: pd.DataFrame) -> pd.DataFrame:
    out = long_df.copy()
    if "WasteFlag" not in out.columns:
        out["WasteFlag"] = derive_waste_flag(out)
    out = out.merge(fold_df, on="Country Code", how="left", validate="m:1")
    out["row_uid"] = (
        out["Country Code"].astype(str)
        + "_"
        + out["Year"].astype(str)
        + "_"
        + out["WasteFlag"].astype(str)
    )
    if out["row_uid"].duplicated().any():
        out["row_uid"] = out["row_uid"] + "_" + out.groupby("row_uid").cumcount().astype(str)
    return out


def build_fold_balance_summary(row_manifest: pd.DataFrame) -> pd.DataFrame:
    d = row_manifest.copy()
    d["log_wg"] = np.log1p(pd.to_numeric(d["Waste_Generation_t"], errors="coerce").clip(lower=0.0))
    summary = (
        d.groupby("outer_fold", dropna=False)
        .agg(
            n_rows=("row_uid", "count"),
            n_countries=("Country Code", "nunique"),
            n_years=("Year", "nunique"),
            wg_mean=("Waste_Generation_t", "mean"),
            wg_median=("Waste_Generation_t", "median"),
            log_wg_mean=("log_wg", "mean"),
        )
        .reset_index()
        .sort_values("outer_fold")
    )
    return summary


def _write_fold_files(row_manifest: pd.DataFrame, out_root: Path):
    for fold in range(1, N_OUTER_FOLDS + 1):
        fold_dir = out_root / f"fold_{fold}"
        raw_dir = fold_dir / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)

        train_df = row_manifest.loc[row_manifest["outer_fold"] != fold].copy()
        test_df = row_manifest.loc[row_manifest["outer_fold"] == fold].copy()
        train_df = train_df.sort_values(["Year", "Country Code", "WasteFlag", "row_uid"]).reset_index(drop=True)
        test_df = test_df.sort_values(["Year", "Country Code", "WasteFlag", "row_uid"]).reset_index(drop=True)

        train_df.insert(0, "Row_ID", np.arange(len(train_df), dtype=int))
        test_df.insert(0, "Row_ID", np.arange(len(test_df), dtype=int))

        train_df.to_csv(raw_dir / "0_train.csv", index=False)
        test_df.to_csv(raw_dir / "0_test.csv", index=False)

        train_df[["Row_ID", "row_uid", "Country Code", "Year", "WasteFlag", "outer_fold"]].to_csv(raw_dir / "0_train_index.csv", index=False)
        test_df[["Row_ID", "row_uid", "Country Code", "Year", "WasteFlag", "outer_fold"]].to_csv(raw_dir / "0_test_index.csv", index=False)

        train_df[["Country Code"]].drop_duplicates().sort_values("Country Code").to_csv(fold_dir / "0_train_country_list.csv", index=False)
        test_df[["Country Code"]].drop_duplicates().sort_values("Country Code").to_csv(fold_dir / "0_test_country_list.csv", index=False)


def run_build(longtable_path: Path, output_root: Path):
    long_df = load_inputs(longtable_path)
    profile = build_country_profile(long_df)
    fold_df = assign_folds_greedy(profile, n_splits=N_OUTER_FOLDS)
    fold_df = enforce_income_waste_coverage(long_df, fold_df, n_splits=N_OUTER_FOLDS)
    row_manifest = build_row_manifest(long_df, fold_df)

    output_root.mkdir(parents=True, exist_ok=True)
    profile.to_csv(output_root / "0_country_profile_table.csv", index=False)
    fold_df.sort_values(["outer_fold", "Country Code"]).to_csv(output_root / "0_country_fold_manifest.csv", index=False)
    row_manifest.to_csv(output_root / "0_row_fold_manifest.csv", index=False)

    balance = build_fold_balance_summary(row_manifest)
    balance.to_csv(output_root / "0_fold_balance_summary.csv", index=False)

    income_fold = fold_df.merge(
        profile[["Country Code", "IncomeGroup", "n_rows"]],
        on="Country Code",
        how="left",
        validate="1:1",
    )
    income_tab = pd.crosstab(income_fold["outer_fold"], income_fold["IncomeGroup"]).reindex(range(1, N_OUTER_FOLDS + 1), fill_value=0)
    row_sum = income_fold.groupby("outer_fold")["n_rows"].sum().reindex(range(1, N_OUTER_FOLDS + 1), fill_value=0.0)
    row_gap = float(row_sum.max() - row_sum.min())
    row_gap_rel = float(row_gap / max(float(row_sum.mean()), 1e-12))
    income_spread = (income_tab.max(axis=0) - income_tab.min(axis=0)).sort_values(ascending=False)

    lines = [
        "# NBCP-CV Fold Balance Report",
        "",
        "This report summarizes outer 5-fold balance under NBCP-CV.",
        "",
        "## Fold Summary",
        "",
        balance.to_markdown(index=False),
        "",
        "## IncomeGroup by Fold (Country Count)",
        "",
        income_tab.to_markdown(),
        "",
        "## Gap Diagnostics",
        "",
        f"- row_gap: {row_gap:.0f}",
        f"- row_gap_rel: {row_gap_rel:.6f}",
        "",
        "### IncomeGroup Spread (max-min)",
        "",
        income_spread.to_frame(name="spread").to_markdown(),
        "",
    ]
    (output_root / "0_fold_balance_report.md").write_text("\n".join(lines), encoding="utf-8")

    _write_fold_files(row_manifest, output_root)


def main() -> None:
    release_root = find_release_root(Path.cwd().resolve())
    step_root = release_root / "Step4_ModelInputs"
    output_dir = step_root / "2_Results"
    input_path = release_root / "Step3_BuildLongTable" / "2_Results" / "0_LongTable_SetABC.csv"
    run_build(input_path, output_dir)
    print(f"observed rows: {pd.read_csv(output_dir / '0_row_fold_manifest.csv').shape[0]}")
    print(f"results root: {output_dir}")


In [ ]:
main()
